# performing ab testing for model comparison

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a baseline classifier with scikit-learn
- Evaluate using accuracy + confusion matrix
- Show how to encode categorical features

## 🔗 Prerequisites

- ✅ Python basics
- ✅ Jupyter Notebook basics

---

## Official Structure Reference

This notebook covers practical activities from **Course 11, Unit 5**:
- performing ab testing for model comparison
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md`

---


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# small synthetic dataset
rng = np.random.default_rng(123)
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
color = rng.choice(['red','green','blue'], size=n)

y = ((x1 + 0.8*x2 + (color == 'red')*0.6 + rng.normal(scale=0.5, size=n)) > 0.2).astype(int)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'color': color, 'y': y})
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
 ('cat', OneHotEncoder(handle_unknown='ignore'), ['color']),
], remainder='passthrough')

clf = Pipeline([
 ('pre', pre),
 ('model', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print('confusion matrix:', confusion_matrix(y_test, y_pred))
print('\nreport:', classification_report(y_test, y_pred))


## 🌍 Real-World Application: A/B Testing ML Models in Production

Before fully deploying a new model, companies like Netflix and Uber test it against the current model using live traffic. Here is the statistical framework they use.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
print("=== A/B Testing for ML Models (Champion-Challenger) ===")
print("Simulating Netflix-style model A/B testing\n")

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Champion model (currently in production)
champion = LogisticRegression(max_iter=1000, random_state=42)
champion.fit(X_train, y_train)

# Challenger models
challengers = {
    'RandomForest':      RandomForestClassifier(n_estimators=50, random_state=42),
    'GradientBoosting':  GradientBoostingClassifier(n_estimators=50, random_state=42),
}

def ab_test(champion, challenger, X_test, y_test, n_users=1000, split=0.1):
    """Simulate routing 10% of traffic to challenger."""
    champion_idx  = np.random.choice(len(X_test), int(n_users*(1-split)), replace=True)
    challenger_idx = np.random.choice(len(X_test), int(n_users*split), replace=True)
    
    champ_acc  = (champion.predict(X_test[champion_idx]) == y_test[champion_idx]).mean()
    chall_acc  = (challenger.predict(X_test[challenger_idx]) == y_test[challenger_idx]).mean()
    
    # Two-proportion z-test
    n1, n2 = len(champion_idx), len(challenger_idx)
    p1, p2 = champ_acc, chall_acc
    p_pool = (p1*n1 + p2*n2) / (n1+n2)
    z = (p1-p2) / np.sqrt(p_pool*(1-p_pool)*(1/n1+1/n2)) if p_pool*(1-p_pool) > 0 else 0
    p_value = 2*(1 - stats.norm.cdf(abs(z)))
    
    return {'champion_acc': champ_acc, 'challenger_acc': chall_acc, 'p_value': p_value, 'significant': p_value < 0.05}

results = {}
for name, challenger in challengers.items():
    challenger.fit(X_train, y_train)
    r = ab_test(champion, challenger, X_test, y_test)
    results[name] = r
    winner = "CHALLENGER WINS" if r['challenger_acc'] > r['champion_acc'] and r['significant'] else "CHAMPION STAYS"
    print(f"  {name:25s}: champion={r['champion_acc']:.3f}, challenger={r['challenger_acc']:.3f}, "
          f"p={r['p_value']:.3f}  → {winner}")

# Plot results
fig, ax = plt.subplots(figsize=(9, 4))
names = ['Champion'] + list(results.keys())
accs  = [LogisticRegression(max_iter=1000,random_state=42).fit(X_train,y_train).score(X_test,y_test)] +         [r['challenger_acc'] for r in results.values()]
colors = ['gold'] + ['green' if r['significant'] and r['challenger_acc']>accs[0] else 'steelblue' for r in results.values()]
ax.bar(names, accs, color=colors)
ax.axhline(accs[0], linestyle='--', color='gold', label='Champion baseline')
ax.set_ylim([0.88, 1.01]); ax.set_ylabel("Accuracy"); ax.set_title("A/B Test Results — Champion vs Challengers")
ax.legend(); plt.tight_layout(); plt.savefig('/tmp/ab_test_results.png', dpi=72)
print("\nA/B test complete. Green bars = statistically significant improvements.")
print("Real-world: Netflix runs 250+ A/B tests simultaneously using this exact framework.")

## 📝 Summary

You learned **A/B testing and canary deployment** for ML models — safely rolling out new models to a fraction of traffic. The champion/challenger pattern lets you compare models on live traffic without full risk. Netflix A/B tests hundreds of recommendation model variants simultaneously.